## Entorno


In [2]:
from bs4 import BeautifulSoup
import requests
import pandas as pd 
import urllib.parse 

## Solicitud HTTP

### Guardar sitio web en variable

In [3]:
website = 'https://www.laencontre.com.pe/alquiler/departamentos/lima/san-martin-porres'

### Solicitud GET

In [4]:
headers = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64)'} # <-- Para evitar que te bloqueen
response = requests.get(website, headers=headers, timeout=(5, 15) )

### Código de Estado

In [5]:
print ("=== INTERPRETACION ===")
print ("403:rechazado" " | " "200:aprobado")
response.status_code

=== INTERPRETACION ===
403:rechazado | 200:aprobado


200

## Objeto Soup

In [6]:
soup = BeautifulSoup(response.content, 'html.parser')

In [7]:
soup


<!DOCTYPE html>
<!--[if gte IE 6]><html class="ie" lang="es_PE"><![endif]--><!--[if !IE]--><html lang="es_PE"><!--[endif]-->
<head>
<meta content="width=device-width, initial-scale=1.0" name="viewport"/><meta content="text/html; charset=utf-8" http-equiv="Content-Type"/><meta content="es_PE" http-equiv="content-language"/><meta content="no-cache" http-equiv="Pragma"/><meta content="no-cache" http-equiv="Cache-Control"/>
<script type="text/javascript">(window.NREUM||(NREUM={})).init={privacy:{cookies_enabled:true},ajax:{deny_list:["bam.nr-data.net"]},distributed_tracing:{enabled:true}};(window.NREUM||(NREUM={})).loader_config={agentID:"1834328805",accountID:"3255554",trustKey:"3255554",xpid:"VwQCVFNWDBAFUlVTDwMOVlE=",licenseKey:"NRJS-49d1957d7e9674a128f",applicationID:"564182905",browserID:"1834328805"};;/*! For license information please see nr-loader-spa-1.324.0.min.js.LICENSE.txt */
(()=>{var e={8122(e,t,r){"use strict";r.d(t,{a:()=>i});var n=r(944);function i(e,t){try{if(!e||"objec

## Resultados

In [8]:
results = soup.find_all('li', {'class':'serp-snippet ad featured'})

In [9]:
# length
len(results)

29

In [10]:
results[1]

<li class="serp-snippet ad featured" data-ev="30316130633534352d373232652d376432302d626362322d6134306132613334393935652d302d312d32" id="01a0c545-722e-7d20-bcb2-a40a2a34995e" itemscope="" itemtype="https://schema.org/Residence">
<div class="ad-ph"> <div class="agencyLogo" itemscope="" itemtype="https://schema.org/RealEstateAgent">
<img alt="GO Grupo Olavarria Bienes Raices" itemprop="logo" src="https://images.proppit.com/publisher/00b652ec-0157-4851-88d1-4557a6b448ea/logo.jpeg"/>
<meta content="GO Grupo Olavarria Bienes Raices" itemprop="name"/>
<meta content="https://images.proppit.com/publisher/00b652ec-0157-4851-88d1-4557a6b448ea/logo.jpeg" itemprop="image"/>
</div>
<div class="slider-ad">
<img alt="Departamento en  San Martin De Porres, Lima" src="https://img.resemmedia.com/eyJidWNrZXQiOiJwcmQtbGlmdWxsY29ubmVjdC1iYWNrZW5kLWIyYi1pbWFnZXMiLCJrZXkiOiJpbmdlc3Rlci8wMWEwYzU0NS03MjJlLTdkMjAtYmNiMi1hNDBhMmEzNDk5NWUvMjJlZTE1MTU0MmE1MzZjZGI3MGI3ZjU4ZDhhNTdjNzFhOTM2MmIxOWQzNGU5ZTA3YTBkMDNiN2E0

## objetivos: Datos Necesarios 

In [11]:
# Name
# Price
# Area
# Rooms
# Bathrooms
# Descryption
# Url

### Nombre
- a porque el nombre esta dentro de un ancla (link): <a
- strip=True --> limpia espacios

In [12]:
results[0].find('a', {'class': 'detail-redirection'}).get_text(strip=True)

'Departamento en  Calle Clemente Palma, Lima, Perú'

### Precio
- div porque el precio esta dentro de una division: <div

In [13]:
results[0].find('div', {'class':'price'}).get_text(strip=True)

'$ 1,200destacado'

### Área
- span porque el metraje esta dentro de una linea corta: <spa

In [14]:
results[0].find('span', {'class':'areaBuilt'}).get_text(strip=True)

'120m2'

### Habitaciones

In [15]:
results[0].find('span', {'class':'rooms'}).get_text(strip=True)

'3'

### Baños

In [16]:
results[0].find('span', {'class':'bathrooms'}).get_text(strip=True)

'2'

### Descripción

In [17]:
results[0].find('p', {'class':'description'}).get_text(strip=True)

'ALQUILO FLAT EN 4TO PISO CON ASCENSOR EN CALLE CLEMENTE - MAGDALENA\nAT: 120 M2 , SE ALQUILA: US$1200 DOLARES\nSALA COMEDOR...'

### URL Relativa

In [18]:
relative_url = results[0].find('a', {'class': 'detail-redirection'}).get('href')

In [19]:
relative_url

'/inmueble/9f5e-b3f4-1a0a6a2-4e50f7399bd8-7626'

In [20]:
root_url = 'https://www.laencontre.com.pe'

In [21]:
# combine URLs
url_combined = root_url + relative_url

In [22]:
url_combined

'https://www.laencontre.com.pe/inmueble/9f5e-b3f4-1a0a6a2-4e50f7399bd8-7626'

## Unir todo dentro de un bucle For (Paginación)

In [23]:
product_name = []
product_price = []
product_area = []
product_rooms = []
product_bathrooms = []
product_description = []
relative_url = []


for result in results:
    
    # name
    try:
        product_name.append(result.find('a', {'class': 'detail-redirection'}).get_text(strip=True)) 
    except:
        product_name.append('n/a') # <--. Si la etiqueta no existe o falla, atrapa el error y agrega 'n/a', luego continua agregando

    # price
    try:
        product_price.append(result.find('div', {'class':'price'}).get_text(strip=True)) 
    except:
        product_price.append('n/a')

    # area
    try:
        product_area.append(result.find('span', {'class':'areaBuilt'}).get_text(strip=True)) 
    except:
        product_area.append('n/a')

    # rooms
    try:
        product_rooms.append(result.find('span', {'class':'rooms'}).get_text(strip=True)) 
    except:
        product_rooms.append('n/a')
        
    # bathrooms
    try:
        product_bathrooms.append(result.find('span', {'class':'bathrooms'}).get_text(strip=True))
    except:
        product_bathrooms.append('n/a')

    #description
    try:
        product_description.append(result.find('p', {'class':'description'}).get_text(strip=True))
    except:
        product_description.append('n/a')


    # relative URL
    try:
        relative_url.append(result.find('a', {'class': 'detail-redirection'}).get('href'))
    except:
        relative_url.append('n/a')
    


### Combinar URLs
- urllib.parse : reconstruye

In [24]:
url_combined = []

for link in relative_url:
    url_combined.append(urllib.parse.urljoin(root_url, link))

## Crear DataFrame de Pandas

In [25]:
product_overview = pd.DataFrame({'Name': product_name, 'Price':product_price, 'Area (m2)':product_area,
                                'Rooms': product_rooms, 'Bathrooms': product_bathrooms, 'Description':product_description,
                                'Link': url_combined})

In [26]:
product_overview.head()

,Name,Price,Area (m2),Rooms,Bathrooms,Description,Link
0,"Departamento en Calle Clemente Palma, Lima, Perú","$ 1,200destacado",120m2,3,2,ALQUILO FLAT EN 4TO PISO CON ASCENSOR EN CALLE...,https://www.laencontre.com.pe/inmueble/9f5e-b3...
1,"Departamento en San Martin De Porres, Lima",$ 377.94destacado,63.1m2,3,2,"¡DEPARTAMENTO EN ESTRENO! Cómodo, Equipado y c...",https://www.laencontre.com.pe/inmueble/72b4-bd...
2,"Departamento en San Martín De Porres, Lima, L...",$ 261.65,58m2,2,1,Imagina vivir en un lugar donde todo te queda ...,https://www.laencontre.com.pe/inmueble/e7dd-a8...
3,"Departamento en San Martin De Porres, Lima",$ 261.65destacado,63m2,2,1,ALQUILO DEPARTAMENTO EN AV. EL AMARGON LOS OLI...,https://www.laencontre.com.pe/inmueble/2072-a9...
4,"Departamento en San Martin De Porres, Lima",$ 206,75m2,1,1,"Acogedor y funcional departamento de 75 m², ub...",https://www.laencontre.com.pe/inmueble/8d67-9f...


## Descarga

In [27]:
product_overview.to_csv('Apartments.csv', index=False)

## Parte 2 - Paginación - Extraer 20 Páginas

In [28]:
product_name = []
product_price = []
product_area = []
product_rooms = []
product_bathrooms = []
product_description = []
relative_url = []

for i in range (1,40):
    
    # website in variable
    if i == 1:
        website = 'https://www.laencontre.com.pe/alquiler/departamentos/lima/san-martin-porres'
    else:
        website = f'https://www.laencontre.com.pe/alquiler/departamentos/lima/san-martin-porres/p_{i}'
    
    # request
    headers = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64)'}
    response = requests.get(website, headers=headers)
    
    # soup object
    soup = BeautifulSoup(response.content, 'html.parser')
   
    # results
    results = soup.find_all('li', {'class':'serp-snippet ad featured'})
    
    # loop through results
    for result in results:
    
        # name
        try:
            product_name.append(result.find('a', {'class': 'detail-redirection'}).get_text(strip=True)) 
        except:
            product_name.append('n/a') # <--. Si la etiqueta no existe o falla, atrapa el error y agrega 'n/a', luego continua agregando

        # price
        try:
            product_price.append(result.find('div', {'class':'price'}).get_text(strip=True)) 
        except:
            product_price.append('n/a')

        # area
        try:
            product_area.append(result.find('span', {'class':'areaBuilt'}).get_text(strip=True)) 
        except:
            product_area.append('n/a')

        # rooms
        try:
            product_rooms.append(result.find('span', {'class':'rooms'}).get_text(strip=True)) 
        except:
            product_rooms.append('n/a')
            
        # bathrooms
        try:
            product_bathrooms.append(result.find('span', {'class':'bathrooms'}).get_text(strip=True))
        except:
            product_bathrooms.append('n/a')

        #description
        try:
            product_description.append(result.find('p', {'class':'description'}).get_text(strip=True))
        except:
            product_description.append('n/a')


        # relative URL
        try:
            relative_url.append(result.find('a', {'class': 'detail-redirection'}).get('href'))
        except:
            relative_url.append('n/a')

In [29]:
url_combined = []

for link in relative_url:
    url_combined.append(urllib.parse.urljoin(root_url, link))

In [30]:
product_overview = pd.DataFrame({'Name': product_name, 'Price':product_price, 'Area (m2)':product_area,
                                'Rooms': product_rooms, 'Bathrooms': product_bathrooms, 'Description':product_description,
                                'Link': url_combined})

In [31]:
product_overview

,Name,Price,Area (m2),Rooms,Bathrooms,Description,Link
0,"Departamento en Calle Clemente Palma, Lima, Perú","$ 1,200destacado",120m2,3,2,ALQUILO FLAT EN 4TO PISO CON ASCENSOR EN CALLE...,https://www.laencontre.com.pe/inmueble/9f5e-b3...
1,"Departamento en San Martin De Porres, Lima",$ 377.94destacado,63.1m2,3,2,"¡DEPARTAMENTO EN ESTRENO! Cómodo, Equipado y c...",https://www.laencontre.com.pe/inmueble/72b4-bd...
2,"Departamento en San Martín De Porres, Lima, L...",$ 261.65,58m2,2,1,Imagina vivir en un lugar donde todo te queda ...,https://www.laencontre.com.pe/inmueble/e7dd-a8...
3,"Departamento en San Martin De Porres, Lima",$ 261.65destacado,63m2,2,1,ALQUILO DEPARTAMENTO EN AV. EL AMARGON LOS OLI...,https://www.laencontre.com.pe/inmueble/2072-a9...
4,"Departamento en San Martin De Porres, Lima",$ 206,75m2,1,1,"Acogedor y funcional departamento de 75 m², ub...",https://www.laencontre.com.pe/inmueble/8d67-9f...
5,"Departamento en San Martin De Porres, Lima",$ 206,75m2,1,1,"Acogedor y funcional departamento de 75 m², ub...",https://www.laencontre.com.pe/inmueble/9706-99...
6,Departamento en Avenida Antúnez De Mayolo 150...,$ 377.94destacado,65m2,2,1,Departamento de 2 Dormitorios en Alquiler – Lo...,https://www.laencontre.com.pe/inmueble/9718-83...
7,Departamento en Avenida Antúnez De Mayolo 150...,$ 348.86destacado,60m2,2,1,Departamento de 2 Dormitorios en Alquiler – Lo...,https://www.laencontre.com.pe/inmueble/a72d-8c...
8,"Departamento en Calle Veintiocho, San Martín ...",$ 290.72destacado,95m2,2,1,ALQUILO DEPARTAMENTO FRENTE A PARQUE EN SAN MA...,https://www.laencontre.com.pe/inmueble/669b-9b...
9,Departamento en Avenida Almirante Miguel Grau...,$ 334.33,60m2,2,1,¿Imaginas despertar cada mañana con luz natura...,https://www.laencontre.com.pe/inmueble/e247-8f...


In [32]:
product_overview['Price'] = (product_overview['Price'].str.split('destacado').str[0].str.strip())
product_overview['Area (m2)'] = (product_overview['Area (m2)'].str.split('m2').str[0].str.strip())

## Descarga

In [33]:
product_overview.to_csv('multiple-apartments-pages.csv', index=False)